This notebook uses the expanded dataset prepared by `preprocessing_refined.ipynb`: train on 2018–2024 and test on the held-out 2025 season. Run refined preprocessing before classification. The data-loading cell checks that both datasets contain the expected seasons.


In [ ]:
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.ml.classification import DecisionTreeClassifier, LogisticRegression, RandomForestClassifier
from pyspark.ml.feature import OneHotEncoder, StringIndexer
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

from classification_helper import assemble, feature_importance, perform, print_results, setup_spark

In [ ]:
spark = setup_spark("Classification_strikeball_noplate")

train = spark.read.parquet("./data/temporal_full_traintest/train")
test = spark.read.parquet("./data/temporal_full_traintest/test")

train_years = {row.game_year for row in train.select("game_year").distinct().collect()}
test_years = {row.game_year for row in test.select("game_year").distinct().collect()}
assert train_years == set(range(2018, 2025)), "Training data must contain the 2018-2024 seasons"
assert test_years == {2025}, "Test data must contain only the held-out 2025 season"

print("Training observations:", train.count())
print("Test observations:", test.count())
train.groupBy("game_year").count().orderBy("game_year").show()
test.groupBy("game_year").count().orderBy("game_year").show()

# labels and zone comparisons must use observed plate coordinates
train = train.filter(F.col("zone_observed"))
test = test.filter(F.col("zone_observed"))


In [ ]:
def add_strike_zone(df):
    ball_radius = 0.12
    half_plate = 17 / 24

    return df.withColumn(
        "in_strike_zone",
        F.when(
            (F.abs(F.col("plate_x")) <= half_plate + ball_radius)
            & (F.col("plate_z") >= F.col("sz_bot") - ball_radius)
            & (F.col("plate_z") <= F.col("sz_top") + ball_radius),
            1,
        ).otherwise(0),
    )

train = add_strike_zone(train)
test = add_strike_zone(test)

In [ ]:
pitch_indexer = StringIndexer(inputCol="pitch_type", outputCol="pitch_type_idx")
pitch_encoder = OneHotEncoder(inputCol="pitch_type_idx", outputCol="pitch_type_ohe")
pitch_indexer_model = pitch_indexer.fit(train)
train = pitch_indexer_model.transform(train)
test = pitch_indexer_model.transform(test)
pitch_encoder_model = pitch_encoder.fit(train)
train = pitch_encoder_model.transform(train)
test = pitch_encoder_model.transform(test)

In [ ]:
target = "in_strike_zone"
numerical = [
    "release_speed", "release_pos_x", "release_pos_y", "release_pos_z",
    "release_spin_rate", "release_extension", "spin_axis",
    "sz_top", "sz_bot", "balls", "strikes", "vx0", "vy0",
    "vz0", "ax", "ay", "az"
]
categorical = ["stand_ohe", "p_throws_ohe", "pitch_type_ohe"]
features = numerical + categorical

train, test = assemble(train, test, features)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        featuresCol="features", labelCol=target, maxIter=100
    ),
    "Decision Tree": DecisionTreeClassifier(
        featuresCol="features", labelCol=target, maxDepth=10, seed=42
    ),
    "Random Forest": RandomForestClassifier(
        featuresCol="features", labelCol=target,
        numTrees=100, maxDepth=10, seed=42
    )
}

In [ ]:
metrics = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]
results = perform(train, test, models, metrics, target)
predictions = {
    name: result["predictions"]
    for name, result in results.items()
}
print_results(results)

In [ ]:
for name, prediction in predictions.items():
    print(f"\n{name}")
    prediction.groupBy(target, "prediction").count().orderBy(target, "prediction").show()

train.groupBy(target).count().orderBy(target).show()
test.groupBy(target).count().orderBy(target).show()

In [ ]:
class_reports = {}
for name, prediction in predictions.items():
    pairs = (
        prediction.select("prediction", target)
        .rdd.map(lambda row: (float(row.prediction), float(row[target])))
    )
    class_metrics = MulticlassMetrics(pairs)
    class_reports[name] = [
        (
            class_metrics.precision(float(label)),
            class_metrics.recall(float(label)),
            class_metrics.fMeasure(float(label)),
        )
        for label in [0.0, 1.0]
    ]

for name, scores in class_reports.items():
    print(f"\n{name}")
    for label, (precision, recall, f1) in enumerate(scores):
        print(
            f"Class {label} | Precision: {precision:.3f} | "
            f"Recall: {recall:.3f} | F1: {f1:.3f}"
        )
    print(f"Macro F1: {sum(score[2] for score in scores) / len(scores):.4f}")

In [ ]:
random_forest = models["Random Forest"].fit(train)
rf_importance = feature_importance(random_forest, train)
print(rf_importance.head(30).to_string(index=False))

top_features = rf_importance.head(20).sort_values("importance")
plt.figure(figsize=(10, 8))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Random Forest - Top 20 Feature Importances")
plt.tight_layout()
plt.show()